In [1]:
import os
import pandas as pd
import sys
from pathlib import Path
import numpy as np
from scipy.ndimage import gaussian_filter1d
import natsort

project_root = Path.cwd().parents[0]
sys.path.append(str(project_root))

from algorithms.tss_file import TSSFile
from algorithms.wfm_edge import WFM_edge

In [2]:
folder = r'C:\Users\FeiyuanYang\OneDrive - Cambridge GaN Devices Ltd\Documents\CGD documents\Projects\P42016\P42016 pump up QFN88'
output_folder = r'C:\Users\FeiyuanYang\OneDrive - Cambridge GaN Devices Ltd\Documents\CGD documents\Projects\P42016'
labels = {
    'Vg': 'ch1',
    'Vgi': 'ch2',
    'Vd': 'ch3',
    'Vdd8': 'ch4',
    'Vghs': 'ch5',
}
files = os.listdir(folder)
tss_files = natsort.natsorted(files)
multifiles = [f for f in files if 'multiFr' in f]
singlefiles = [f for f in files if 'multiFr' not in f]

In [3]:
output = [['Board ID', 'Variant', 'Vdc', 'Fsw (kHz)', 'dt (ns)', 'Vg', 'Rgon', 'data type', 'temperature (C)', 
           'Pulse number', 'Vdd8_settled_lsoff', 
]]

In [4]:
for f in multifiles:
    """if f != 'HB1_BY71_0Vd_65kHz_400ns_15Vg_33R2Rg_RT_multiFr.tss':
        continue"""
    names = f.split('_')
    tss = TSSFile(folder, f)
    time = tss.waveforms[labels['Vg']].time_for_frame()
    frame = tss.waveforms[labels['Vg']].frame_count
    for i in range(frame):
        vg = tss.waveforms[labels['Vg']].values_for_frame(i)
        vdd8 = tss.waveforms[labels['Vdd8']].values_for_frame(i)
        try:
            vg_edges = WFM_edge(vg, time, rising_edge_number=2, falling_edge_number=1, peak_width=1, peak_distance=10, sigma=100)
            idx_lson = vg_edges.get_edge(2, "rising", 1)['thresh_idx']
            idx_lsoff = vg_edges.get_edge(1, "falling", 1)['thresh_idx']
            #print(i, idx_lson, idx_lsoff)
            # vdd8_max = max(vdd8[idx_lsoff:idx_lson])
            vdd8_settled = np.average(vdd8[(idx_lson + idx_lsoff) // 2:idx_lson])
        except:
            vdd8_max = 0
            vdd8_settled = 0
            print(f)

        line = [names[0], names[1], names[2][:-2], names[3][:-3], names[4][:-2], names[5][:-2], names[6][:-4], names[7] , names[8][:-4],
                i + 1, vdd8_settled]
        output.append(line)

c:\Users\FeiyuanYang\OneDrive - Cambridge GaN Devices Ltd\Documents\CGD_codes\DP_Algorithm-main\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:571: RuntimeWarning: Mean of empty slice.
  avg = a.mean(axis, **keepdims_kw)
c:\Users\FeiyuanYang\OneDrive - Cambridge GaN Devices Ltd\Documents\CGD_codes\DP_Algorithm-main\.venv\Lib\site-packages\numpy\_core\_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [5]:
for f in singlefiles:
    """if f != 'HB5_LU72_400Vd_500kHz_100ns_15Vg_2R2Rg_RT.tss':
        continue"""
    names = f.split('_')
    tss = TSSFile(folder, f)
    time = tss.waveforms[labels['Vg']].time_for_frame()
    vg = tss.waveforms[labels['Vg']].values_for_frame(0)
    vdd8 = tss.waveforms[labels['Vdd8']].values_for_frame(0)
    vg_edges = WFM_edge(vg, time, rising_edge_number=10000, falling_edge_number=10000, peak_width=1, peak_distance=10, sigma=100)
    #print(vg_edges.falling_edge_number)
    #print(vg_edges.rising_edge_number)
    for i in range(1, vg_edges.falling_edge_number):
        try:
            idx_lson = vg_edges.get_edge(i+1, "rising", 1)['thresh_idx']
            idx_lsoff = vg_edges.get_edge(i, "falling", 1)['thresh_idx']
            #vdd8_max = max(vdd8[idx_lson:idx_lsoff])
            vdd8_settled = np.average(vdd8[(idx_lson + idx_lsoff) // 2:idx_lson])

        except:
            vdd8_max = 0
            vdd8_settled = 0
        line = [names[0], names[1], names[2][:-2], names[3][:-3], names[4][:-2], names[5][:-2], names[6][:-4], names[7] , 'SingleFr',
                i, vdd8_settled]
        output.append(line)
        #print(i, vdd8_max, vdd8_settled)

In [6]:
output_df = pd.DataFrame(output)
output_path = os.path.join(output_folder, 'P42016_Vdd8_pump_up_lsoff.csv')
output_df.to_csv(output_path, index=False, header=False)